In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
from IPython.display import display, Markdown

# Semana 9B — Detección de Anomalías: Métodos Estadísticos (Laboratorio)

**Objetivo:** Aplicar z-score, MAD e IQR al dataset ClimaLab,
compararlos sistemáticamente y entender cuándo cada uno falla o acierta.

**Ejercicios:**

1. Reglas físicas (línea base)
2. Z-score global sobre `tdb`
3. Z-score estacional
4. MAD e IQR sobre `p_atm`
5. Comparación integrada

In [ ]:
df = pd.read_parquet("data/ClimaLab_2023-05-31_2025-06-20.parquet")

---
## 1. Reglas físicas (línea base)

Antes de aplicar métodos estadísticos, marcamos valores que son
**físicamente imposibles** — no se necesita estadística, solo
conocimiento del dominio.

In [ ]:
reglas_fisicas = {
    "rh < 0": df["rh"] < 0,
    "p_atm < 500 o > 1100": (df["p_atm"] < 500) | (df["p_atm"] > 1100),
    "tdb < -10 o > 50": (df["tdb"] < -10) | (df["tdb"] > 50),
    "ghi > 5 cuando solar_alt < -5°": (df["ghi"] > 5) & (df["solar_altitude"] < -5),
}

outlier_fisico = reglas_fisicas["rh < 0"].copy()
for mask in reglas_fisicas.values():
    outlier_fisico = outlier_fisico | mask

_rows = ""
for nombre, mask in reglas_fisicas.items():
    _rows += f"| {nombre} | {mask.sum():,} |\n"
_rows += f"| **Total (unión)** | **{outlier_fisico.sum():,}** |"

display(Markdown(f"""
| Regla | # Registros |
|:---|---:|
{_rows}

Estos son **outliers seguros** — nuestra línea base para evaluar
los métodos estadísticos.
"""))

---
## 2. Z-score global sobre `tdb`

Calculamos z sobre **toda** la serie y marcamos $|z| > 3$.

**Problema esperado:** Temperaturas normales de verano/invierno
serán marcadas como outliers porque el z-score global no distingue
estaciones.

In [ ]:
tdb = df["tdb"].dropna()
_mu = tdb.mean()
_sigma = tdb.std()
z_global = (tdb - _mu) / _sigma
out_z_global = np.abs(z_global) > 3

fig2, ax2 = plt.subplots(figsize=(13, 4))

# Subsample para visualización
_step = max(1, len(tdb) // 50000)
_idx = tdb.index[::_step]
_vals = tdb.values[::_step]
_out_mask = out_z_global.values[::_step]

ax2.scatter(_idx[~_out_mask], _vals[~_out_mask],
            c="steelblue", s=1, alpha=0.2)
ax2.scatter(_idx[_out_mask], _vals[_out_mask],
            c="crimson", s=5, zorder=5, alpha=0.5)
ax2.axhline(_mu + 3 * _sigma, color="crimson", ls="--", lw=1,
            label=f"+3σ = {_mu + 3 * _sigma:.1f} °C")
ax2.axhline(_mu - 3 * _sigma, color="crimson", ls="--", lw=1,
            label=f"−3σ = {_mu - 3 * _sigma:.1f} °C")
ax2.set_ylabel("tdb (°C)")
ax2.set_title(
    f"Z-score global sobre tdb — {out_z_global.sum():,} detecciones"
)
ax2.legend()
plt.tight_layout()
plt.show()

---
## 3. Z-score estacional (hora × mes)

Agrupamos `tdb` por **(hora del día, mes)** y calculamos z dentro de
cada grupo.  Así, "35°C a las 3PM en julio" se compara solo con
otros registros de las 3PM en julio.

In [ ]:
grupos = tdb.groupby([tdb.index.hour, tdb.index.month])
media_grupo = grupos.transform("mean")
std_grupo = grupos.transform("std")
z_estacional = (tdb - media_grupo) / std_grupo
z_estacional = z_estacional.fillna(0)
out_z_estac = np.abs(z_estacional) > 3

In [ ]:
fig3, ax3 = plt.subplots(figsize=(13, 4))
step = max(1, len(tdb) // 50000)
idx_sub = tdb.index[::step]
vals_sub = tdb.values[::step]
out_sub = out_z_estac.values[::step]
ax3.scatter(idx_sub[~out_sub], vals_sub[~out_sub],
            c="steelblue", s=1, alpha=0.2)
ax3.scatter(idx_sub[out_sub], vals_sub[out_sub],
            c="darkorange", s=5, zorder=5, alpha=0.5)
ax3.set_ylabel("tdb (°C)")
ax3.set_title(
    f"Z-score estacional (hora × mes) — {out_z_estac.sum():,} detecciones"
)
plt.tight_layout()
plt.show()

In [ ]:
n_global = out_z_global.sum()
n_estac = out_z_estac.sum()
display(Markdown(f"""
> **Comparación:**
> - Z-score global: **{n_global:,}** detecciones
>   (incluye falsos positivos estacionales)
> - Z-score estacional: **{n_estac:,}** detecciones
>   (solo valores anómalos *dentro de su contexto temporal*)
>
> La reducción drástica indica que muchas "anomalías" del z-score
> global eran simplemente valores normales de verano o invierno.
"""))

---
## 4. MAD e IQR sobre `p_atm`

La presión atmosférica tiene outliers extremos (443 hPa) que inflan
σ.  Comparamos cómo reaccionan los tres métodos.

In [ ]:
patm = df["p_atm"].dropna()

# Z-score
_mu = patm.mean()
_sigma = patm.std()
out_patm_z = np.abs((patm - _mu) / _sigma) > 3

# MAD
_mediana = patm.median()
_mad = np.median(np.abs(patm - _mediana))
_mad_scaled = 1.4826 * _mad
out_patm_mad = np.abs(patm - _mediana) / _mad_scaled > 3

# IQR
_q1 = patm.quantile(0.25)
_q3 = patm.quantile(0.75)
_iqr = _q3 - _q1
out_patm_iqr = (patm < _q1 - 1.5 * _iqr) | (patm > _q3 + 1.5 * _iqr)

display(Markdown(f"""
### Estadísticos de `p_atm`

| Estadístico | Valor |
|:---|---:|
| μ | {_mu:.2f} hPa |
| σ | {_sigma:.2f} hPa |
| Mediana | {_mediana:.2f} hPa |
| MAD × 1.4826 | {_mad_scaled:.2f} hPa |
| Q1, Q3 | {_q1:.2f}, {_q3:.2f} hPa |
| IQR | {_iqr:.2f} hPa |

### Detecciones

| Método | # Outliers |
|:---|---:|
| Z-score (±3σ) | {out_patm_z.sum():,} |
| MAD (±3·MAD) | {out_patm_mad.sum():,} |
| IQR (Tukey 1.5) | {out_patm_iqr.sum():,} |
"""))

In [ ]:
fig4, axes4 = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

_configs = [
    (out_patm_z, "Z-score", "crimson"),
    (out_patm_mad, "MAD", "darkorange"),
    (out_patm_iqr, "IQR", "seagreen"),
]

_step = max(1, len(patm) // 50000)
_idx = patm.index[::_step]
_vals = patm.values[::_step]

for ax, (out_mask, nombre, color) in zip(axes4, _configs):
    _om = out_mask.values[::_step]
    ax.scatter(_idx[~_om], _vals[~_om], c="steelblue", s=1, alpha=0.2)
    ax.scatter(_idx[_om], _vals[_om], c=color, s=8, zorder=5, alpha=0.6)
    ax.set_ylabel("p_atm (hPa)")
    ax.set_title(f"{nombre}: {out_mask.sum():,} detecciones")

axes4[2].set_xlabel("Fecha")
plt.suptitle("Comparación de métodos sobre p_atm", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

> **Observa:** σ está inflado por los outliers extremos, así que
> el z-score tiene un umbral demasiado amplio y detecta menos.
> MAD e IQR son inmunes a la inflación y detectan más anomalías.

---
## 5. Comparación integrada sobre `rh`

Aplicamos los tres métodos (z-score estacional, MAD, IQR) a la
humedad relativa y comparamos cuántos outliers detecta cada uno
y cuántos coinciden.

In [ ]:
rh = df["rh"].dropna()

# Z-score estacional (groupby vectorizado para evitar for-loops)
rh_grupos = rh.groupby([rh.index.hour, rh.index.month])
rh_media = rh_grupos.transform("mean")
rh_std = rh_grupos.transform("std")
rh_z_estac = ((rh - rh_media) / rh_std).fillna(0)
out_rh_z = np.abs(rh_z_estac) > 3

# MAD
med_rh = rh.median()
mad_rh = np.median(np.abs(rh - med_rh))
mad_s_rh = 1.4826 * mad_rh
out_rh_mad = np.abs(rh - med_rh) / mad_s_rh > 3

# IQR
q1_rh, q3_rh = rh.quantile(0.25), rh.quantile(0.75)
iqr_rh = q3_rh - q1_rh
out_rh_iqr = (rh < q1_rh - 1.5 * iqr_rh) | (rh > q3_rh + 1.5 * iqr_rh)

# Tabla de coincidencias
comparacion = pd.DataFrame({
    "z_estacional": out_rh_z,
    "mad": out_rh_mad,
    "iqr": out_rh_iqr,
})
comparacion["alguno"] = comparacion.any(axis=1)
comparacion["todos"] = comparacion[["z_estacional", "mad", "iqr"]].all(axis=1)

In [ ]:
_solo_z = (comparacion["z_estacional"] & ~comparacion["mad"] & ~comparacion["iqr"]).sum()
_solo_mad = (~comparacion["z_estacional"] & comparacion["mad"] & ~comparacion["iqr"]).sum()
_solo_iqr = (~comparacion["z_estacional"] & ~comparacion["mad"] & comparacion["iqr"]).sum()

display(Markdown(f"""
### Conteos de detección sobre `rh`

| Categoría | # Registros |
|:---|---:|
| Z-score estacional | {comparacion["z_estacional"].sum():,} |
| MAD | {comparacion["mad"].sum():,} |
| IQR | {comparacion["iqr"].sum():,} |
| Detectado por **algún** método | {comparacion["alguno"].sum():,} |
| Detectado por **todos** | {comparacion["todos"].sum():,} |
| Solo z-score estacional | {_solo_z:,} |
| Solo MAD | {_solo_mad:,} |
| Solo IQR | {_solo_iqr:,} |
"""))

In [ ]:
fig5, ax5 = plt.subplots(figsize=(13, 5))

_step = max(1, len(rh) // 50000)
_idx = rh.index[::_step]
_vals = rh.values[::_step]

# Base
ax5.scatter(_idx, _vals, c="steelblue", s=1, alpha=0.15, label="Normal")

# Colorear por método
_oz = out_rh_z.values[::_step]
_om = out_rh_mad.values[::_step]
_oi = out_rh_iqr.values[::_step]

ax5.scatter(_idx[_oz], _vals[_oz], c="crimson", s=6, alpha=0.5,
            label=f"Z-estacional ({out_rh_z.sum():,})")
ax5.scatter(_idx[_om & ~_oz], _vals[_om & ~_oz],
            c="darkorange", s=6, alpha=0.5,
            label=f"Solo MAD ({(out_rh_mad & ~out_rh_z).sum():,})")
ax5.scatter(_idx[_oi & ~_oz & ~_om], _vals[_oi & ~_oz & ~_om],
            c="seagreen", s=6, alpha=0.5,
            label=f"Solo IQR ({(out_rh_iqr & ~out_rh_z & ~out_rh_mad).sum():,})")

ax5.set_ylabel("rh (%)")
ax5.set_xlabel("Fecha")
ax5.set_title("Outliers en rh — coloreados por método que los detectó")
ax5.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

> **Discusión:**
> - Los outliers detectados por **todos** los métodos son los más
>   confiables — probablemente son errores reales de sensor.
> - Los detectados por **solo un** método merecen inspección manual:
>   ¿son falsos positivos del método más agresivo o anomalías
>   sutiles que los otros métodos no capturan?
> - En la **semana 10** añadiremos STL e Isolation Forest para
>   completar el arsenal de detección.

---
## Resumen de la sesión

| Ejercicio | Hallazgo principal |
|:---|:---|
| **Reglas físicas** | Línea base: `rh < 0`, `p_atm < 500` son errores seguros sin necesidad de estadística |
| **Z-score global** | Falla con datos estacionales — marca valores normales de verano/invierno como outliers |
| **Z-score estacional** | Agrupar por (hora, mes) elimina falsos positivos estacionales |
| **MAD e IQR vs z-score** | MAD e IQR son robustos a outliers extremos que inflan σ (caso `p_atm`) |
| **Comparación** | Los tres métodos coinciden en los outliers más severos; los desacuerdos requieren juicio |

**Próxima semana (10):** Detección basada en descomposición (STL)
e Isolation Forest multivariado.